In [1]:
import pandas as pd
import numpy as np

ARCHIVO = "rotacion_inventario_base_dashboard_odoo_autoazur.xlsx"
HOJA = "ventas_conjunto_detalle"
DIAS_ANALISIS_3M = 90

# -----------------------------
# Carga
# -----------------------------
df = pd.read_excel(ARCHIVO, sheet_name=HOJA)
df.columns = [str(c).strip() for c in df.columns]

# Validaciones mínimas
for col in ["fecha", "cantidad", "sku_madre", "canal"]:
    if col not in df.columns:
        raise ValueError(f"Falta la columna obligatoria: {col}")

# -----------------------------
# Limpieza
# -----------------------------
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")
df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)

# Si existe la bandera de match, nos quedamos con ventas vinculadas
if "tiene_referencia_madre" in df.columns:
    df = df[df["tiene_referencia_madre"].astype(str).str.upper().eq("SI")].copy()

# Solo filas con SKU madre válido
df = df[
    df["sku_madre"].notna()
    & df["sku_madre"].astype(str).str.strip().ne("")
    & df["fecha"].notna()
].copy()

# -----------------------------
# Últimos 3 meses
# -----------------------------
fecha_fin = df["fecha"].max().normalize()
fecha_inicio = fecha_fin - pd.Timedelta(days=DIAS_ANALISIS_3M - 1)

df_3m = df[(df["fecha"] >= fecha_inicio) & (df["fecha"] <= fecha_fin + pd.Timedelta(days=1))].copy()

# -----------------------------
# Resumen por SKU madre y canal
# -----------------------------
resumen = (
    df_3m.groupby(["sku_madre", "canal"], as_index=False)
    .agg(ventas_canal=("cantidad", "sum"))
)

# Total por SKU madre
resumen["ventas_totales_sku"] = resumen.groupby("sku_madre")["ventas_canal"].transform("sum")

# Porcentaje por canal
resumen["porcentaje_canal"] = (
    resumen["ventas_canal"] / resumen["ventas_totales_sku"] * 100
)

# Ordenar
resumen = resumen.sort_values(
    ["sku_madre", "ventas_canal"],
    ascending=[True, False]
).reset_index(drop=True)

# -----------------------------
# Pivot opcional: una fila por SKU madre
# -----------------------------
pivot_pct = (
    resumen.pivot_table(
        index="sku_madre",
        columns="canal",
        values="porcentaje_canal",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

pivot_ventas = (
    resumen.pivot_table(
        index="sku_madre",
        columns="canal",
        values="ventas_canal",
        fill_value=0,
        aggfunc="sum"
    )
    .reset_index()
)

# -----------------------------
# Exportar
# -----------------------------
salida = "ventas_por_canal_ultimos_3m.xlsx"
with pd.ExcelWriter(salida, engine="openpyxl") as writer:
    resumen.to_excel(writer, sheet_name="detalle", index=False)
    pivot_pct.to_excel(writer, sheet_name="porcentajes", index=False)
    pivot_ventas.to_excel(writer, sheet_name="unidades", index=False)

print(f"Archivo generado: {salida}")
print(resumen.head(20))

Archivo generado: ventas_por_canal_ultimos_3m.xlsx
   sku_madre    canal  ventas_canal  ventas_totales_sku  porcentaje_canal
0     IQ1004     Full             1                   1        100.000000
1     IQ1005     Full           183                 186         98.387097
2     IQ1005     Drop             2                 186          1.075269
3     IQ1005  ELEKTRA             1                 186          0.537634
4     IQ1006     Full            22                  22        100.000000
5     IQ1007     Drop             5                   5        100.000000
6     IQ1009     Full             6                   6        100.000000
7     IQ1019     Full             6                   6        100.000000
8      IQ102     Full           484                 487         99.383984
9      IQ102   AMAZON             3                 487          0.616016
10    IQ1021     Full             2                   2        100.000000
11    IQ1022     Full             9                   9      